In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pinn import PINN, SIREN, pde_loss, bc_loss, u_exact

In [2]:
def sample_interior(N):
    pts = torch.rand(N, 2)
    pts.requires_grad_(True)
    return pts


def sample_boundary(N):
    assert N % 4 == 0, f"N must be divisible by 4, got {N}"
    n = N // 4
    # bottom: y=0, x in [0,1]
    bottom = torch.stack([torch.rand(n), torch.zeros(n)], dim=1)
    # top: y=1
    top = torch.stack([torch.rand(n), torch.ones(n)], dim=1)
    # left: x=0
    left = torch.stack([torch.zeros(n), torch.rand(n)], dim=1)
    # right: x=1
    right = torch.stack([torch.ones(n), torch.rand(n)], dim=1)
    return torch.cat([bottom, top, left, right], dim=0)

In [3]:
N_test = 100
xy_int = sample_interior(N_test)
xy_bc = sample_boundary(N_test)
assert xy_int.shape == (N_test, 2) and xy_int.requires_grad
assert xy_bc.shape == (N_test, 2)
# All boundary points should be on edge
assert ((xy_bc[:, 0] == 0) | (xy_bc[:, 0] == 1) |
        (xy_bc[:, 1] == 0) | (xy_bc[:, 1] == 1)).all()
print("Семплирование: ОК")

Семплирование: ОК


In [4]:
torch.manual_seed(42)

model_pinn = PINN(hidden_layers=4, hidden_width=64)
model_siren = SIREN(hidden_layers=4, hidden_width=64, omega_0=30)

optimizer_pinn = torch.optim.Adam(model_pinn.parameters(), lr=1e-3)
optimizer_siren = torch.optim.Adam(model_siren.parameters(), lr=1e-4)

n_params_pinn = sum(p.numel() for p in model_pinn.parameters())
n_params_siren = sum(p.numel() for p in model_siren.parameters())
print(f"PINN параметров: {n_params_pinn}")
print(f"SIREN параметров: {n_params_siren}")

PINN параметров: 12737
SIREN параметров: 12737


In [5]:
N_col = 2000
N_bc = 500
n_epochs = 5000
lambda_bc = 10
log_every = 500

xy_col_train = sample_interior(N_col)
xy_bc_train = sample_boundary(N_bc)

history_pinn = {'pde': [], 'bc': [], 'total': []}
history_siren = {'pde': [], 'bc': [], 'total': []}

for epoch in range(1, n_epochs + 1):
    xy_col_train.grad = None

    # --- PINN ---
    optimizer_pinn.zero_grad()
    loss_pde_p = pde_loss(model_pinn, xy_col_train)
    loss_bc_p = bc_loss(model_pinn, xy_bc_train)
    loss_p = loss_pde_p + lambda_bc * loss_bc_p
    loss_p.backward()
    optimizer_pinn.step()

    # --- SIREN ---
    optimizer_siren.zero_grad()
    loss_pde_s = pde_loss(model_siren, xy_col_train)
    loss_bc_s = bc_loss(model_siren, xy_bc_train)
    loss_s = loss_pde_s + lambda_bc * loss_bc_s
    loss_s.backward()
    optimizer_siren.step()

    history_pinn['pde'].append(loss_pde_p.item())
    history_pinn['bc'].append(loss_bc_p.item())
    history_pinn['total'].append(loss_p.item())
    history_siren['pde'].append(loss_pde_s.item())
    history_siren['bc'].append(loss_bc_s.item())
    history_siren['total'].append(loss_s.item())

    if epoch % log_every == 0:
        print(f"Эпоха {epoch:5d} | PINN: {loss_p.item():.4e} | SIREN: {loss_s.item():.4e}")

print("Обучение завершено")


Эпоха   500 | PINN: 1.8236e+01 | SIREN: 1.6020e+00
Эпоха  1000 | PINN: 1.4742e+00 | SIREN: 9.1683e-01


KeyboardInterrupt: 

In [ ]:
epochs = np.arange(1, n_epochs + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

titles = ['Суммарные потери', 'PDE потери', 'BC потери']
keys = ['total', 'pde', 'bc']

for ax, title, key in zip(axes, titles, keys):
    ax.semilogy(epochs, history_pinn[key], color='steelblue', label='PINN (tanh)')
    ax.semilogy(epochs, history_siren[key], color='darkorange', label='SIREN (sin)')
    ax.set_title(title)
    ax.set_xlabel('Эпоха')
    ax.set_ylabel('Потери (лог. шкала)')
    ax.legend()
    ax.grid(True, which='both', ls='--')

plt.tight_layout()
plt.show()


In [ ]:
model_pinn.eval()
model_siren.eval()

N_vis = 100
x_lin = torch.linspace(0, 1, N_vis)
y_lin = torch.linspace(0, 1, N_vis)
XX, YY = torch.meshgrid(x_lin, y_lin, indexing='ij')
xy_vis = torch.stack([XX.flatten(), YY.flatten()], dim=1)

with torch.no_grad():
    u_true = u_exact(xy_vis[:, 0], xy_vis[:, 1]).reshape(N_vis, N_vis).numpy()
    u_pred_pinn = model_pinn(xy_vis).squeeze().reshape(N_vis, N_vis).numpy()
    u_pred_siren = model_siren(xy_vis).squeeze().reshape(N_vis, N_vis).numpy()

err_pinn = np.abs(u_pred_pinn - u_true)
err_siren = np.abs(u_pred_siren - u_true)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

plots = [
    (u_true,        'Точное решение u_exact'),
    (u_pred_pinn,   'PINN предсказание'),
    (u_pred_siren,  'SIREN предсказание'),
    (err_pinn,      '|Ошибка| PINN'),
    (err_siren,     '|Ошибка| SIREN'),
]

for idx, (data, title) in enumerate(plots):
    ax = axes[idx // 3, idx % 3]
    im = ax.imshow(data.T, origin='lower', extent=[0, 1, 0, 1],
                   cmap='viridis', aspect='equal')
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    plt.colorbar(im, ax=ax)

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
def relative_l2(u_pred, u_true):
    return np.linalg.norm(u_pred - u_true) / np.linalg.norm(u_true)

l2_pinn = relative_l2(u_pred_pinn, u_true)
l2_siren = relative_l2(u_pred_siren, u_true)

df = pd.DataFrame({
    'Модель': ['PINN (tanh)', 'SIREN (sin)'],
    'Относительная ошибка L2': [f'{l2_pinn:.4e}', f'{l2_siren:.4e}'],
    'Лучше': ['✓' if l2_pinn < l2_siren else '', '✓' if l2_siren < l2_pinn else ''],
})
print(df.to_string(index=False))